# Lesson 1 notebook — attention and Gaussian diffusion

This notebook checks two derivations from `notes.md`: scaled dot-product attention and the closed-form DDPM forward marginal. Run top to bottom, then change the dimensions and noise schedule.

## 1. Stable scaled dot-product attention

For `Q: (n_query, d_k)`, `K: (n_key, d_k)`, and `V: (n_key, d_v)`, form `Q @ K.T / sqrt(d_k)`, normalize each query row over keys, and take the weighted sum of values.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def softmax(x, axis=-1):
    x = np.asarray(x, dtype=float)
    e = np.exp(x - np.max(x, axis=axis, keepdims=True))
    return e / e.sum(axis=axis, keepdims=True)

def attention(Q, K, V, causal=False):
    Q, K, V = (np.asarray(x, dtype=float) for x in (Q, K, V))
    if Q.ndim != 2 or K.ndim != 2 or V.ndim != 2:
        raise ValueError('Q, K, V must be rank-2 arrays')
    if Q.shape[1] != K.shape[1] or K.shape[0] != V.shape[0]:
        raise ValueError('incompatible query/key dimensions or key/value lengths')
    logits = Q @ K.T / np.sqrt(Q.shape[1])
    if causal:
        if Q.shape[0] != K.shape[0]:
            raise ValueError('causal self-attention requires equal query and key lengths')
        logits = np.where(np.triu(np.ones_like(logits, dtype=bool), k=1), -np.inf, logits)
    weights = softmax(logits, axis=-1)
    return weights @ V, weights, logits


In [ ]:
rng = np.random.default_rng(17)
n, d_k, d_v = 5, 4, 3
Q, K, V = rng.normal(size=(n,d_k)), rng.normal(size=(n,d_k)), rng.normal(size=(n,d_v))
out, weights, logits = attention(Q, K, V)
assert logits.shape == weights.shape == (n,n) and out.shape == (n,d_v)
assert np.allclose(weights.sum(axis=1), 1) and np.all(weights >= 0)
assert np.allclose(softmax(logits + 123), weights)  # shift invariance
_, causal_weights, _ = attention(Q, K, V, causal=True)
assert np.allclose(causal_weights.sum(axis=1), 1)
assert np.all(causal_weights[np.triu_indices(n, k=1)] == 0)
print('Q,K,V -> output:', Q.shape, K.shape, V.shape, '->', out.shape)
print('Full attention weights:\n', weights.round(3))
print('Causal weights (future positions zero):\n', causal_weights.round(3))
fig, ax = plt.subplots(1,2, figsize=(8,3), constrained_layout=True)
ax[0].imshow(weights, vmin=0, vmax=1); ax[0].set_title('Full attention')
ax[1].imshow(causal_weights, vmin=0, vmax=1); ax[1].set_title('Causal attention')
plt.show()


## 2. The forward diffusion marginal

A transition `x_t = sqrt(alpha_t) x_(t-1) + sqrt(1-alpha_t) epsilon_t` has a closed-form marginal. Signal coefficients multiply; independent Gaussian noise covariances add.

In [ ]:
def forward_chain(x0, alphas, rng):
    states = [np.asarray(x0, dtype=float)]
    x = states[0].copy()
    for alpha in alphas:
        x = np.sqrt(alpha)*x + np.sqrt(1-alpha)*rng.normal(size=x.shape)
        states.append(x)
    return np.stack(states)

def direct_marginal(x0, alphas, n_samples, rng):
    a_bar = np.prod(alphas)
    eps = rng.normal(size=(n_samples, *np.shape(x0)))
    return np.sqrt(a_bar)*x0 + np.sqrt(1-a_bar)*eps

alphas = np.array([0.91, 0.84, 0.79, 0.88, 0.93])
a_bar = np.cumprod(alphas)
x0, t, n_samples = 1.25, 4, 100_000
recursive = np.array([forward_chain(x0, alphas[:t], np.random.default_rng(i))[-1] for i in range(n_samples)])
direct = direct_marginal(x0, alphas[:t], n_samples, np.random.default_rng(100_001))
mu, var = np.sqrt(a_bar[t-1])*x0, 1-a_bar[t-1]
print('analytic mean, variance:', mu, var)
print('recursive empirical:', recursive.mean(), recursive.var())
print('direct empirical:   ', direct.mean(), direct.var())
assert abs(recursive.mean()-mu) < .015 and abs(direct.mean()-mu) < .015
assert abs(recursive.var()-var) < .015 and abs(direct.var()-var) < .015
xs = np.linspace(mu-4*np.sqrt(var), mu+4*np.sqrt(var), 300)
pdf = np.exp(-.5*((xs-mu)**2)/var)/np.sqrt(2*np.pi*var)
plt.hist(recursive, bins=65, density=True, alpha=.4, label='recursive chain')
plt.hist(direct, bins=65, density=True, alpha=.4, label='direct marginal')
plt.plot(xs, pdf, 'k-', lw=2, label='analytic Gaussian')
plt.xlabel('$x_t$'); plt.ylabel('density'); plt.legend(); plt.show()


## 3. Vector state and covariance check

For isotropic noise, the covariance is `(1-alpha_bar) I`. The empirical matrix fluctuates around this target.

In [ ]:
x0_vec = np.array([1.0, -0.5, 2.0])
samples = direct_marginal(x0_vec, alphas, 50_000, np.random.default_rng(4))
mean_exact = np.sqrt(np.prod(alphas))*x0_vec
cov_exact = (1-np.prod(alphas))*np.eye(x0_vec.size)
print('empirical mean:', samples.mean(axis=0).round(3))
print('exact mean:    ', mean_exact.round(3))
print('empirical covariance:\n', np.cov(samples, rowvar=False).round(3))
print('exact covariance:\n', cov_exact.round(3))
assert np.allclose(samples.mean(axis=0), mean_exact, atol=.025)
assert np.allclose(np.cov(samples, rowvar=False), cov_exact, atol=.025)


## Reflection

1. Which results are exact identities, and which checks are statistical approximations?
2. Why do recursive and direct samples have matching distributions but different realized values?
3. What changes if each transition has a general noise covariance matrix?

Use the solutions in `../exercises.md` only after attempting these questions.